In [ ]:
import pandas as pd
import numpy as np
import cupy as cp
import os
import glob
import warnings
from xgboost import XGBRFRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')

# ==========================================
# 统一配置接口
# ==========================================
INPUT_DIR = "./engineered_features"
OUTPUT_CSV = "Progressive_LOYO_Overall_Results.csv"

# 特征排除接口：在此填入需要强制剔除的特征名称
# 建议优先剔除 'Cum_GDD_P1' 以观察精度演变曲线的修正效果
EXCLUDE_FEATURES = [
    'Cum_GDD_P1',
    'Cum_GDD_P2',
    'Cum_GDD_P3',
    'Cum_GDD_P4',
    'Cum_GDD_P5'
    # 'Cum_Precip_P1', # 可按需取消注释或继续添加
]
# ==========================================

def calculate_metrics(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mean_true = np.mean(y_true)
    rrmse = (rmse / mean_true) * 100 
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100 
    
    den = np.sum((np.abs(y_pred - mean_true) + np.abs(y_true - mean_true)) ** 2)
    d_index = 1 - (np.sum((y_pred - y_true) ** 2) / den) if den != 0 else 0
    
    return {
        'R2': round(r2, 3), 'RMSE': round(rmse, 3), 'RRMSE(%)': round(rrmse, 3),
        'MAE': round(mae, 3), 'MAPE(%)': round(mape, 3), 'd-index': round(d_index, 3)
    }

def run_progressive_training(input_dir):
    csv_files = glob.glob(os.path.join(input_dir, "*.csv"))
    if not csv_files:
        print("未找到 CSV 文件，请检查路径。")
        return
        
    df = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
    
    # 仅保留部分静态特征
    static_features = ['Sand', 'Clay', 'SOC']
    meta_cols = ['Year', 'Zone', 'latitude', 'longitude', 'yield']
    all_columns = df.columns.tolist()
    
    periods = ['P1', 'P2', 'P3', 'P4', 'P5']
    all_overall_results = []
    
    # 固定的 XGBoost 随机森林超参数
    fixed_params = {
        'n_estimators': 150,
        'max_depth': 10,
        'colsample_bynode': 0.4,
        'subsample': 0.8,
        'random_state': 42,
        'n_jobs': -1,
        'tree_method': 'hist',
        'device': 'cuda'
    }
    
    years = sorted(df['Year'].unique())

    for i in range(len(periods)):
        current_stage = periods[i]
        active_periods = periods[:i+1]
        
        # 构建动态特征列表，并执行黑名单过滤
        dynamic_features = [
            col for col in all_columns 
            if any(p in col for p in active_periods) 
            and col not in meta_cols 
            and col not in static_features
            and col not in EXCLUDE_FEATURES
        ]
        current_features = static_features + dynamic_features
        
        print(f"\n{'='*70}")
        print(f">>> 阶段: {current_stage} (包含: {' + '.join(active_periods)}) | 特征数: {len(current_features)}")
        print(f"{'='*70}")
        
        all_y_true, all_y_pred = [], []
        fold_importances = []
        
        # 执行 LOYO 交叉验证
        for test_year in years:
            train_df = df[df['Year'] != test_year]
            test_df = df[df['Year'] == test_year]
            
            X_train_np = train_df[current_features].values
            y_train_np = train_df['yield'].values
            X_test_np = test_df[current_features].values
            y_test_np = test_df['yield'].values
            
            X_train_gpu = cp.array(X_train_np)
            y_train_gpu = cp.array(y_train_np)
            X_test_gpu = cp.array(X_test_np)
            
            rf = XGBRFRegressor(**fixed_params)
            rf.fit(X_train_gpu, y_train_gpu)
            
            y_pred_gpu = cp.asarray(rf.predict(X_test_gpu))
            y_pred = cp.asnumpy(y_pred_gpu)
            
            all_y_true.extend(y_test_np)
            all_y_pred.extend(y_pred)
            
            fold_importances.append(rf.feature_importances_)
            
            m = calculate_metrics(y_test_np, y_pred)
            # 输出单年份指标，补全 RRMSE 和 MAPE 及其他所有核心指标
            print(f"    [Year {test_year}] R2: {m['R2']:.3f} | RMSE: {m['RMSE']:.2f} | RRMSE: {m['RRMSE(%)']:.2f}% | MAE: {m['MAE']:.2f} | MAPE: {m['MAPE(%)']:.2f}% | d-index: {m['d-index']:.3f}")

        # 计算本阶段汇总指标
        global_metrics = calculate_metrics(all_y_true, all_y_pred)
        global_metrics['Stage'] = current_stage
        global_metrics['Test_Year'] = 'Overall'
        all_overall_results.append(global_metrics)
        
        # 输出阶段总体汇总指标
        m = global_metrics
        print(f"\n    -> [{current_stage} 总体汇总] R2: {m['R2']:.3f} | RMSE: {m['RMSE']:.2f} | RRMSE: {m['RRMSE(%)']:.2f}% | MAE: {m['MAE']:.2f} | MAPE: {m['MAPE(%)']:.2f}% | d-index: {m['d-index']:.3f}")

        # 汇总并输出排名前 5 的特征
        avg_importances = np.mean(fold_importances, axis=0)
        importance_df = pd.DataFrame({
            'Feature': current_features,
            'Importance': avg_importances
        }).sort_values(by='Importance', ascending=False)
        
        top_5_features = importance_df.head(5)
        print(f"    -> [{current_stage} 平均排名前 5 的特征]:")
        for idx, row in top_5_features.iterrows():
            print(f"       * {row['Feature']}: {row['Importance']:.4f}")

    # 保存总体结果至 CSV
    cols = ['Stage', 'Test_Year', 'R2', 'RRMSE(%)', 'd-index', 'MAPE(%)', 'RMSE', 'MAE']
    results_df = pd.DataFrame(all_overall_results)[cols]
    results_df.to_csv(OUTPUT_CSV, index=False)
    print(f"\n所有阶段执行完毕。汇总总体指标已保存至: {OUTPUT_CSV}")

if __name__ == "__main__":
    run_progressive_training(INPUT_DIR)


>>> 阶段: P1 (包含: P1) | 特征数: 25
    [Year 2016] R2: 0.214 | RMSE: 841.64 | RRMSE: 13.56% | MAE: 571.49 | MAPE: 9.42% | d-index: 0.625
    [Year 2017] R2: -0.058 | RMSE: 997.91 | RRMSE: 15.70% | MAE: 821.45 | MAPE: 13.41% | d-index: 0.534
    [Year 2018] R2: 0.341 | RMSE: 769.47 | RRMSE: 12.80% | MAE: 565.01 | MAPE: 10.28% | d-index: 0.687
    [Year 2019] R2: 0.261 | RMSE: 832.39 | RRMSE: 13.46% | MAE: 602.69 | MAPE: 10.67% | d-index: 0.623
    [Year 2020] R2: 0.299 | RMSE: 657.35 | RRMSE: 10.77% | MAE: 501.12 | MAPE: 9.00% | d-index: 0.684
    [Year 2021] R2: 0.398 | RMSE: 690.88 | RRMSE: 11.37% | MAE: 547.42 | MAPE: 9.54% | d-index: 0.768

    -> [P1 总体汇总] R2: 0.245 | RMSE: 806.06 | RRMSE: 13.09% | MAE: 601.54 | MAPE: 10.39% | d-index: 0.648
    -> [P1 平均排名前 5 的特征]:
       * P1_GDD: 0.1700
       * P1_Fertility_Vigor: 0.1029
       * P1_VPD_max: 0.0863
       * Cum_VPD_P1: 0.0695
       * P1_VPD: 0.0654

>>> 阶段: P2 (包含: P1 + P2) | 特征数: 50
    [Year 2016] R2: 0.225 | RMSE: 835.91 | RRMS